<div dir="rtl">

# جلسه ۳: بازبینی، دیباگ و تست کد تولیدشده با هوش مصنوعی

**Reviewing, Debugging and Testing AI-Generated Code**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/03.%20Reviewing_and_Testing_AI_Code.ipynb)

**دوره مهندسی LLM و هوش مصنوعی مولد** — کیوان محمودی

### اهداف این جلسه

- شناختن خطاهای رایج کد تولیدشده با هوش مصنوعی
- نوشتن تست‌های پارامتری با pytest برای کشف موارد مرزی، به‌خصوص در متن فارسی
- تشخیص APIهای ساختگی و منسوخ و راستی‌آزمایی آن‌ها
- پیدا کردن و رفع آسیب‌پذیری‌های امنیتی رایج: SQL Injection، کلید در کد و eval
- بازبینی تست‌هایی که خود هوش مصنوعی نوشته است

</div>

<div dir="rtl">

## ۱. چرا کد هوش مصنوعی باید بازبینی شود؟

مدل‌های زبانی کدی تولید می‌کنند که **باورپذیر** است: تمیز، با توضیحات و ظاهراً درست. ولی مدل کد را اجرا نکرده، نیازهای واقعی شما را نمی‌داند و الگوهای رایج اینترنت، شامل الگوهای غلط و قدیمی، را تکرار می‌کند. هر کدی که در پروژه شما قرار می‌گیرد، مسئولیتش با شماست، نه با ابزار.

### خطاهای رایج

| دسته | نمونه |
|---|---|
| موارد مرزی | ورودی خالی، ارقام فارسی، فاصله‌های اضافه، مقدار صفر |
| API ساختگی یا منسوخ | فراخوانی تابعی که وجود ندارد یا در نسخه جدید حذف شده |
| امنیت | SQL Injection، کلید API در کد، `eval` روی ورودی کاربر |
| منطق | شرط‌های اشتباه، off-by-one، فرض‌های پنهان درباره داده |
| اضافه‌کاری | قابلیت‌ها و وابستگی‌هایی که کسی نخواسته |
| تست‌های گمراه‌کننده | تست‌هایی که رفتار غلط را «تأیید» می‌کنند |

در این جلسه هر دسته را با یک نمونه واقعی بررسی می‌کنیم. این نوت‌بوک به مدل زبانی نیاز ندارد. کدها نمونه‌هایی شبیه خروجی واقعی ابزارهای هوش مصنوعی‌اند.

### چک‌لیست بازبینی

1. آیا هر خط را می‌فهمم؟
2. ورودی‌های مرزی و نامعتبر چه می‌شوند؟
3. همه توابع و کتابخانه‌ها واقعاً وجود دارند و نسخه درست است؟
4. داده کاربر کجا وارد کوئری، دستور سیستم یا `eval` می‌شود؟
5. رازها (کلید، رمز) کجا نگهداری می‌شوند؟
6. تست‌ها رفتار درست را بررسی می‌کنند یا فقط رفتار فعلی را؟

</div>

<div dir="rtl">

## ۲. مطالعه موردی ۱: موارد مرزی

از یک ابزار خواسته‌ایم «تابعی بنویس که شماره موبایل ایرانی را به قالب 09xxxxxxxxx تبدیل کند». این پاسخ را گرفته‌ایم:

</div>

In [ ]:
%%writefile phone.py
import re


def normalize_mobile(phone: str) -> str:
    """Normalise an Iranian mobile number to the form 09xxxxxxxxx."""
    digits = re.sub(r"\D", "", phone)
    if digits.startswith("98"):
        digits = "0" + digits[2:]
    if len(digits) == 11:
        return digits
    raise ValueError(f"invalid mobile number: {phone}")

<div dir="rtl">

کد کوتاه و تمیز است و برای `09123456789` درست کار می‌کند. ولی شماره‌ها در دنیای واقعی به شکل‌های مختلف وارد می‌شوند. به جای حدس زدن، **فهرست ورودی‌های واقعی** را به شکل تست می‌نویسیم. `parametrize` به ما اجازه می‌دهد ده‌ها ورودی را با یک تابع تست بررسی کنیم.

</div>

In [ ]:
%%writefile test_phone.py
import pytest
from phone import normalize_mobile


@pytest.mark.parametrize("raw, expected", [
    ("09123456789", "09123456789"),
    ("0912 345 6789", "09123456789"),
    ("+98 912 345 6789", "09123456789"),
    ("00989123456789", "09123456789"),
    ("9123456789", "09123456789"),
    ("۰۹۱۲۳۴۵۶۷۸۹", "09123456789"),
    ("۰۹۱۲-۳۴۵-۶۷۸۹", "09123456789"),
])
def test_valid_numbers(raw, expected):
    assert normalize_mobile(raw) == expected


@pytest.mark.parametrize("raw", [
    "02112345678",      # landline, not a mobile number
    "0912345678",       # one digit short
    "",
    "شماره ندارم",
])
def test_invalid_numbers(raw):
    with pytest.raises(ValueError):
        normalize_mobile(raw)

In [ ]:
!python -m pytest -q --tb=line test_phone.py

<div dir="rtl">

گزارش pytest دقیقاً نشان می‌دهد کدام ورودی‌ها شکست خورده‌اند. اشکال‌ها:

- **ارقام فارسی:** `\D` در پایتون ارقام یونیکد را هم «رقم» حساب می‌کند. بنابراین `re.sub` آن‌ها را حذف نمی‌کند و رشته خروجی شامل ارقام فارسی می‌ماند. پایتون ۳ به‌صورت پیش‌فرض یونیکد را می‌فهمد و این گاهی غافلگیرکننده است.
- **پیشوند 0098:** فقط `98` بررسی می‌شود.
- **شماره بدون صفر:** `9123456789` رد می‌شود.
- **تلفن ثابت:** `02112345678` یازده رقم است و پذیرفته می‌شود، در حالی که شماره موبایل نیست.

نسخه اصلاح‌شده:

</div>

In [ ]:
%%writefile phone.py
import re

_PERSIAN_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize_mobile(phone: str) -> str:
    """Normalise an Iranian mobile number to the form 09xxxxxxxxx.

    Accepts Persian, Arabic-Indic or English digits, spaces, dashes and the
    prefixes +98, 0098 and 98. Raises ValueError for anything that is not a
    mobile number.
    """
    digits = re.sub(r"\D", "", phone.translate(_PERSIAN_DIGITS))
    for prefix in ("0098", "98"):
        if digits.startswith(prefix) and len(digits) == len(prefix) + 10:
            digits = digits[len(prefix):]
            break
    if len(digits) == 10 and digits.startswith("9"):
        digits = "0" + digits
    if len(digits) == 11 and digits.startswith("09"):
        return digits
    raise ValueError(f"invalid mobile number: {phone!r}")

In [ ]:
!python -m pytest -q test_phone.py 2>&1 | tail -n 3

<div dir="rtl">

**درس:** ارزش اصلی در تست‌ها بود، نه در کد. با این تست‌ها می‌توانید هر نسخه دیگری را هم، چه خودتان بنویسید چه ابزار، بلافاصله بسنجید.

### موارد مرزی رایج در متن فارسی

| مورد | نمونه | چرا مشکل‌ساز است |
|---|---|---|
| ارقام فارسی و عربی | ۱۲۳ و ١٢٣ | کدپوینت‌های متفاوت از 123 |
| ی و ک عربی | علي، كتاب | ظاهر یکسان، رشته متفاوت؛ جستجو شکست می‌خورد |
| نیم‌فاصله (ZWNJ) | می‌روم و می روم | `split()` و مقایسه رشته‌ها را تحت تأثیر قرار می‌دهد |
| ممیز و جداکننده فارسی | ۱۲٫۵ و ۱۲٬۰۰۰ | `float()` آن‌ها را نمی‌شناسد |
| علائم جهت‌نما | U+200F و U+200E | نامرئی‌اند ولی رشته را تغییر می‌دهند |

</div>

In [ ]:
pairs = [("علی", "علي"), ("کتاب", "كتاب"), ("می‌روم", "می روم")]
for a, b in pairs:
    print(f"{a!r} == {b!r}: {a == b}   code points: {[hex(ord(c)) for c in a]} vs {[hex(ord(c)) for c in b]}")

try:
    float("۱۲٫۵")
except ValueError as e:
    print("\nfloat('۱۲٫۵') ->", e)
print("int('۱۲۳') ->", int("۱۲۳"), " (int accepts Persian digits, float does not accept '٫')")

<div dir="rtl">

## ۳. مطالعه موردی ۲: API ساختگی و منسوخ

مدل‌ها روی کدهای قدیمی‌تر آموزش دیده‌اند و گاهی تابعی را پیشنهاد می‌کنند که حذف شده یا اصلاً وجود نداشته است. به این پدیده **توهم (Hallucination)** می‌گوییم. نمونه رایج: متد `DataFrame.append` که در pandas ۲ حذف شد، ولی مدل‌ها هنوز زیاد پیشنهادش می‌دهند.

</div>

In [ ]:
import pandas as pd

print("pandas", pd.__version__)
orders = pd.DataFrame({"product": ["P1001"], "qty": [1]})

# Code suggested by an assistant:
try:
    orders = orders.append({"product": "P2001", "qty": 2}, ignore_index=True)
except AttributeError as e:
    print("AttributeError:", e)

# The current API:
orders = pd.concat([orders, pd.DataFrame([{"product": "P2001", "qty": 2}])], ignore_index=True)
print(orders)

<div dir="rtl">

**راستی‌آزمایی:** هر تابع ناآشنا را قبل از پذیرش بررسی کنید. سریع‌ترین راه‌ها:

</div>

In [ ]:
!pip -q install hazm

import importlib.metadata
import inspect

import hazm

print("installed hazm version:", importlib.metadata.version("hazm"))
print("Normalizer has 'normalize':", hasattr(hazm.Normalizer, "normalize"))
print("Normalizer has 'normalize_digits':", hasattr(hazm.Normalizer, "normalize_digits"))
print(inspect.signature(hazm.Normalizer.__init__))

<div dir="rtl">

روش‌های دیگر: `help(تابع)` در پایتون، مستندات رسمی برای **نسخه نصب‌شده** و دادن نسخه کتابخانه‌ها به ابزار در فایل قواعد پروژه. یک خطر جدی‌تر هم وجود دارد: مدل نام بسته‌ای را پیشنهاد می‌دهد که وجود ندارد و مهاجمان گاهی بسته‌ای با همان نام در PyPI منتشر می‌کنند (**Slopsquatting**). قبل از `pip install` بسته ناآشنا، صفحه آن را در PyPI ببینید.

## ۴. مطالعه موردی ۳: امنیت

از ابزار خواسته‌ایم «تابعی بنویس که کالاها را بر اساس نام در دیتابیس جستجو کند». پاسخ:

</div>

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE products (id TEXT, name TEXT, price INTEGER, internal_cost INTEGER)")
db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", [
    ("P1001", "گوشی سامسونگ A55", 28_900_000, 24_000_000),
    ("P1003", "گوشی شیائومی Redmi Note 14", 21_400_000, 17_500_000),
    ("P2001", "لپ‌تاپ لنوو IdeaPad", 41_500_000, 36_000_000),
])


def search_products_unsafe(term):
    query = f"SELECT id, name, price FROM products WHERE name LIKE '%{term}%'"
    return db.execute(query).fetchall()


print(search_products_unsafe("گوشی"))

<div dir="rtl">

کار می‌کند، ولی ورودی کاربر مستقیماً داخل متن کوئری قرار گرفته است. کاربری که به جای نام کالا یک تکه SQL وارد کند، می‌تواند منطق کوئری را عوض کند و اطلاعات محرمانه، مثل قیمت تمام‌شده، را بخواند. به این حمله **تزریق SQL (SQL Injection)** می‌گوییم.

</div>

In [ ]:
attack = "' UNION SELECT id, name, internal_cost FROM products --"
for row in search_products_unsafe(attack):
    print(row)

<div dir="rtl">

راه درست، **کوئری پارامتری** است. مقدار جداگانه به دیتابیس داده می‌شود و هرگز به‌عنوان کد SQL تفسیر نمی‌شود:

</div>

In [ ]:
def search_products(term):
    return db.execute("SELECT id, name, price FROM products WHERE name LIKE ?",
                      (f"%{term}%",)).fetchall()


print(search_products("گوشی"))
print(search_products(attack))

<div dir="rtl">

### سه الگوی خطرناک دیگر

| الگو | چرا خطرناک است | جایگزین |
|---|---|---|
| `api_key = "sk-..."` در کد | با اولین push به GitHub لو می‌رود | متغیر محیطی یا Colab Secrets |
| `eval(user_input)` | اجرای هر کدی که کاربر بفرستد | `ast.literal_eval` یا تجزیه صریح |
| `subprocess.run(cmd, shell=True)` با ورودی کاربر | تزریق دستور سیستم‌عامل | فهرست آرگومان‌ها بدون `shell=True` |

ماشین‌حساب زیر نمونه‌ای از کدی است که ابزارها زیاد تولید می‌کنند:

</div>

In [ ]:
def calculator_unsafe(expression):
    return eval(expression)


print(calculator_unsafe("2 * (3 + 4)"))
print(calculator_unsafe("__import__('os').getcwd()"))   # any Python code runs here

In [ ]:
import ast
import operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculator(expression):
    """Evaluate an arithmetic expression safely (numbers and + - * / ** only)."""
    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](evaluate(node.operand))
        raise ValueError(f"unsupported expression: {expression!r}")
    return evaluate(ast.parse(expression, mode="eval").body)


print(calculator("2 * (3 + 4)"))
try:
    calculator("__import__('os').getcwd()")
except ValueError as e:
    print("ValueError:", e)

<div dir="rtl">

این ماشین‌حساب امن را در جلسه ۱۲ به‌عنوان ابزار در اختیار مدل زبانی قرار می‌دهیم. آنجا ورودی را خود مدل تولید می‌کند و امنیت حتی مهم‌تر می‌شود.

## ۵. مطالعه موردی ۴: تست‌هایی که هوش مصنوعی نوشته

از ابزار خواستن «برای این تابع تست بنویس» کار رایج و مفیدی است، ولی یک دام دارد. ابزار تست را از روی **کد فعلی** می‌نویسد، نه از روی **رفتار درست**. اگر کد باگ داشته باشد، تست باگ را «قفل» می‌کند. نمونه:

</div>

In [ ]:
def apply_discount(price, percent):
    """Apply a percentage discount and round to the nearest 1,000 toman."""
    return round(price * (1 - percent / 100), -3)


# Tests generated from the code above:
def test_apply_discount():
    assert apply_discount(1_000_000, 10) == 900_000
    assert apply_discount(250_000, 15) == 212_000
    assert apply_discount(500_000, 150) == -250_000


test_apply_discount()
print("all generated tests pass")

<div dir="rtl">

همه تست‌ها پاس می‌شوند، ولی آخری یک باگ را «تأیید» کرده است: تخفیف ۱۵۰ درصد قیمت منفی می‌دهد. تستی که بر اساس قواعد کسب‌وکار نوشته شود این را می‌گیرد. همچنین `round` در پایتون از **گرد کردن بانکی** استفاده می‌کند، یعنی ۲۱۲٬۵۰۰ به ۲۱۲٬۰۰۰ گرد می‌شود، نه ۲۱۳٬۰۰۰. آیا این رفتار مطلوب کسب‌وکار است؟ این سؤالی است که فقط انسان می‌تواند جواب دهد.

**قاعده:** تست‌های تولیدشده را مثل کد بازبینی کنید و برای هر تابع دست‌کم چند تست را از روی **قواعد** بنویسید، نه از روی کد.

</div>

In [ ]:
import pytest


def test_discount_rules():
    assert apply_discount(1_000_000, 0) == 1_000_000
    assert apply_discount(1_000_000, 100) == 0
    with pytest.raises(ValueError):
        apply_discount(1_000_000, 150)      # a discount above 100% must be rejected


try:
    test_discount_rules()
except BaseException as e:
    print("rule-based test failed:", type(e).__name__)

<div dir="rtl">

## ۶. دیباگ با کمک هوش مصنوعی

وقتی خطایی می‌بینید، برای کمک گرفتن از ابزار این‌ها را بدهید:

1. **پیام خطای کامل** همراه با traceback، نه خلاصه آن.
2. **کد مرتبط** و نه کل پروژه.
3. **ورودی‌ای که خطا را ایجاد می‌کند** و خروجی مورد انتظار.
4. **آنچه امتحان کرده‌اید.**

و یک درخواست مهم: «**اول علت را توضیح بده، بعد اصلاح کن.**» اگر توضیح قانع‌کننده نبود، اصلاح هم احتمالاً سطحی است و فقط علامت را پنهان می‌کند. نمونه رایج اصلاح سطحی، `try/except` است که خطا را می‌بلعد.

## ۷. جمع‌بندی

- کد هوش مصنوعی باورپذیر است، نه لزوماً درست. مسئولیت با شماست.
- تست‌های پارامتری سریع‌ترین راه کشف موارد مرزی‌اند. متن فارسی موارد مرزی ویژه خودش را دارد.
- هر API ناآشنا را با نسخه نصب‌شده راستی‌آزمایی کنید و بسته ناشناخته نصب نکنید.
- ورودی کاربر هرگز نباید مستقیماً وارد SQL، `eval` یا دستور سیستم‌عامل شود.
- تست‌هایی که ابزار از روی کد می‌نویسد ممکن است باگ را قفل کنند. چند تست را از روی قواعد بنویسید.

## ۸. تمرین

۱. تابع `parse_price` جلسه قبل را با تست‌های جدید محک بزنید: «۱٫۲۵۰٫۰۰۰ تومان»، رشته خالی، «٤٥ هزار تومان» با ارقام عربی و عددی با علامت جهت‌نمای نامرئی `\u200f`. اشکال‌ها را پیدا و رفع کنید.

۲. تابع `apply_discount` را طوری اصلاح کنید که تست `test_discount_rules` پاس شود و گرد کردن همیشه به سمت بالا باشد. تست مناسب برای گرد کردن هم بنویسید.

۳. از یک ابزار هوش مصنوعی بخواهید «تابعی بنویس که نام فایلی را از کاربر بگیرد و محتوایش را از پوشه `uploads` برگرداند». کد را از نظر امنیتی بررسی کنید. با ورودی `../../etc/passwd` چه می‌شود؟ (این حمله **Path Traversal** نام دارد.)

۴. یک تابع ده‌خطی از پروژه‌های قبلی خودتان را به ابزار بدهید و تست بخواهید. چند تست از روی قواعد اضافه کنید که ابزار نوشته بود؟

</div>